In [1]:
#importing libraries
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score,
    recall_score,
    matthews_corrcoef,
    make_scorer
)

In [4]:
#loading dataset
df = pd.read_csv("yeast.csv") 
df.head()

,mcg,gvh,alm,mit,erl,pox,vac,nuc,name
0,0.58,0.61,0.47,0.13,0.5,0.0,0.48,0.22,MIT
1,0.43,0.67,0.48,0.27,0.5,0.0,0.53,0.22,MIT
2,0.64,0.62,0.49,0.15,0.5,0.0,0.53,0.22,MIT
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22,NUC
4,0.42,0.44,0.48,0.54,0.5,0.0,0.48,0.22,MIT


In [5]:
#EDA
df.info()
df.describe()
df["name"].value_counts()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1484 entries, 0 to 1483
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   mcg     1484 non-null   float64
 1   gvh     1484 non-null   float64
 2   alm     1484 non-null   float64
 3   mit     1484 non-null   float64
 4   erl     1484 non-null   float64
 5   pox     1484 non-null   float64
 6   vac     1484 non-null   float64
 7   nuc     1484 non-null   float64
 8   name    1484 non-null   object 
dtypes: float64(8), object(1)
memory usage: 104.5+ KB


mcg     0
gvh     0
alm     0
mit     0
erl     0
pox     0
vac     0
nuc     0
name    0
dtype: int64

In [8]:
#separating features and target
X = df.drop(columns=["name"])
y = df["name"]

In [9]:
#Encoding target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [10]:
#re-checking feature types
X.dtypes

mcg    float64
gvh    float64
alm    float64
mit    float64
erl    float64
pox    float64
vac    float64
nuc    float64
dtype: object

In [11]:
#decision tree
dt = DecisionTreeClassifier(
    criterion="entropy",   
    random_state=42
)


In [12]:
#cross validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [13]:
#F1 Score
f1 = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(f1_score, average="macro")
)

print("F1 Score (macro):", f1.mean())


F1 Score (macro): 0.40109448185954044


In [14]:
#recall
recall = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(recall_score, average="macro")
)

print("Recall (macro):", recall.mean())


Recall (macro): 0.40686597645768485


In [15]:
#MCC
mcc = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(matthews_corrcoef)
)

print("MCC:", mcc.mean())


MCC: 0.36260526129215565


In [17]:
#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
param_grid = {
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

grid = GridSearchCV(
    DecisionTreeClassifier(criterion="entropy", random_state=42),
    param_grid,
    cv=skf,
    scoring="f1_macro"
)

grid.fit(X, y_encoded)

print("Best parameters:", grid.best_params_)

Best parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10}


In [18]:
best_dt = grid.best_estimator_


In [19]:
final_f1 = cross_val_score(
    best_dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(f1_score, average="macro")
)

print("Final F1:", final_f1.mean())


Final F1: 0.42956744740891584


In [20]:
#AUC
auc = cross_val_score(
    best_dt, X, y_encoded,
    cv=skf,
    scoring="roc_auc_ovr"
)

print("AUC:", auc.mean())

AUC: 0.7513039460902352
